# Lesson 5 — Four Knights: Scotch Variation (as Black)
**Project 1500 | Priority #5**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

> **Sample size warning:** This opening has very few games. Stats are directional only.

In [ ]:
import chess
import chess.pgn
import chess.svg
import json
import glob
import io
import re
from collections import Counter, defaultdict
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE, flipped=True):
    svg = chess.svg.board(board, arrows=arrows or [], size=size, flipped=flipped)
    if caption:
        display(HTML(f'<p style="font-family:monospace;font-size:13px;margin:4px 0 8px 0;color:#aaa">{caption}</p>'))
    display(SVG(data=svg))

## Step 1 — Load rapid games as black (April 2025+)

In [ ]:
def load_rapid_games_as_black(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(glob.glob(f'{games_dir}/2025_*.json') + glob.glob(f'{games_dir}/2026_*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            black = g.get('black', {})
            if black.get('username', '').lower() != username.lower(): continue
            result = black.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_black_games = load_rapid_games_as_black(USERNAME)
print(f'Loaded {len(all_black_games)} rapid games as black')

## Step 2 — Filter to Four Knights Scotch games and compute stats

We also detect whether black responded to 4.d4 with **exd4** (correct) or **d6** (passive).

In [ ]:
# Four Knights Scotch: white plays d4 in the Four Knights position
# Detect by position: white has knights on f3+c3, pawns on e4+d4, black has Nc6+Nf6
def is_four_knights_scotch(board):
    """True when white has Nf3, Nc3, pawns on e4 and d4 just played."""
    wf3 = board.piece_at(chess.F3)
    wc3 = board.piece_at(chess.C3)
    we4 = board.piece_at(chess.E4)
    wd4 = board.piece_at(chess.D4)
    return (
        wf3 and wf3.piece_type == chess.KNIGHT and wf3.color == chess.WHITE and
        wc3 and wc3.piece_type == chess.KNIGHT and wc3.color == chess.WHITE and
        we4 and we4.piece_type == chess.PAWN   and we4.color == chess.WHITE and
        wd4 and wd4.piece_type == chess.PAWN   and wd4.color == chess.WHITE
    )

fks_games = []
exd4_games = []
d6_games   = []

for g in all_black_games:
    try:
        game  = chess.pgn.read_game(io.StringIO(g['pgn']))
        board = game.board()
        moves = list(game.mainline_moves())
        detected = False
        for i, move in enumerate(moves[:14]):
            board.push(move)
            if not detected and i % 2 == 0 and is_four_knights_scotch(board):
                detected = True
                fks_games.append(g)
                # Next black move is the response to d4
                if i + 1 < len(moves):
                    board2 = board.copy()
                    resp = moves[i + 1]
                    san = board2.san(resp)
                    if san == 'exd4':
                        exd4_games.append(g)
                    elif san == 'd6':
                        d6_games.append(g)
                break
    except Exception:
        pass

counts = Counter(g['outcome'] for g in fks_games)
total  = len(fks_games)
wr     = 100 * counts['win'] / total if total else 0

print(f'Four Knights Scotch games as black: {total}')
print(f'W:{counts["win"]}  L:{counts["loss"]}  D:{counts["draw"]}  Win%: {wr:.1f}%')
print(f'\nResponses detected:')
print(f'  exd4 (correct): {len(exd4_games)} games')
print(f'  d6   (passive): {len(d6_games)} games')
print()
if total < 10:
    print('Sample too small for reliable junction stats. Lesson is principle-based.')

## Step 3 — Board diagrams *(illustrative example lines)*

In [ ]:
# [ILLUSTRATIVE] The Four Knights Scotch position — after 4.d4
# Verified legal: e4 e5 Nf3 Nc6 Nc3 Nf6 d4
AFTER_D4 = ['e4','e5','Nf3','Nc6','Nc3','Nf6','d4']

show_board(
    board_after(AFTER_D4),
    arrows=[
        chess.svg.Arrow(chess.E5, chess.D4, color='#27ae60'),  # exd4 — take the pawn
        chess.svg.Arrow(chess.D7, chess.D6, color='#c0392b'),  # d6 — passive
    ],
    caption='[ILLUSTRATIVE] After 4.d4 — Green=exd4 (take the pawn, correct) | Red=d6 (passive, gives white d5)'
)

In [ ]:
# [ILLUSTRATIVE] Why d6 fails — white pushes d5 and black is cramped
# Verified legal: after d4 d6, white plays d5 and black's Nc6 is awkward
D6_LINE = ['e4','e5','Nf3','Nc6','Nc3','Nf6','d4','d6','d5','Nb8']

show_board(
    board_after(D6_LINE),
    arrows=[
        chess.svg.Arrow(chess.B8, chess.B8, color='#c0392b'),  # highlight retreated knight
        chess.svg.Arrow(chess.D5, chess.D5, color='#c0392b'),  # white space advantage
    ],
    caption='[ILLUSTRATIVE] After d6 d5 Nb8 — knight retreated, white has huge space advantage'
)

In [ ]:
# [ILLUSTRATIVE] The correct response — exd4 Nxd4 Bc5, active play
# Verified legal: e4 e5 Nf3 Nc6 Nc3 Nf6 d4 exd4 Nxd4 Bc5
CORRECT_LINE = ['e4','e5','Nf3','Nc6','Nc3','Nf6','d4','exd4','Nxd4','Bc5']

show_board(
    board_after(CORRECT_LINE),
    arrows=[
        chess.svg.Arrow(chess.C5, chess.D4, color='#27ae60'),  # Bc5 attacks the knight
        chess.svg.Arrow(chess.C6, chess.E5, color='#2980b9'),  # Nc6-e5 plan
    ],
    caption='[ILLUSTRATIVE] After exd4 Nxd4 Bc5 — bishop attacks the knight on d4, black is active'
)

## Step 4 — Summary

In [ ]:
print('FOUR KNIGHTS SCOTCH — SUMMARY')
print('=' * 50)
print(f'\nTotal games detected: {total}  ({wr:.1f}% win rate)')
if total < 10:
    print('(Small sample — focus on the principle, not the percentages)')
print()
print('RULES:')
print('  1. After 4.d4 — ALWAYS play exd4, take the pawn')
print('  2. NEVER play d6 — white gets a free d5 push and you are cramped')
print('  3. After exd4 Nxd4 — play Bc5 (attacks knight) or Bb4 (pins Nc3)')